# Taking a data packet: step-by-step walkthrough of `take_packet.py`

This notebook goes through **every step** of the packet acquisition script so you can check it
before it is ever run on the DAQ machine.

**Nothing here touches the hardware.** The real script sends text to the cdaq queue. Here the same
text is run on *fake boards* (`src/fake_daq.py`). The fake boards copy old real files, so the
processing notebook has realistic content to work on.

### What one packet is

Each board has **its own** parameter set: θ (new) and θ_ref (reference).

| # | what | where | why |
|---|---|---|---|
| 1 | go to θ_ref | each board, its own values | start every packet from the reference |
| 2 | **3 flushes** | all boards | empty the CCDs, so every packet starts from the same state |
| 3 | **3 reference trace sets**, a flush between sets (set = 4 channels, every traced board) | standalone mode | the amplifiers' reference state *now*; 3 repeats give the run-to-run scatter |
| 4 | go to θ, then a flush | each board, its own values | the change |
| 5 | **3 trace sets** at θ, a flush between sets | standalone mode | the cheap measurement at the new settings |
| 6 | **one image** at θ, of the size given in `[image]`, then back to standalone and a flush | all boards together, master/slave, **one queue entry** | the expensive ground truth: WADERs gives σe, gain and λ |
| 7 | **manifest** (JSON) | written by the script | links the files with θ, θ_ref, Δθ, the previous packet, and every step's receipt |

### Files

| file | role |
|---|---|
| `src/take_packet.py` | plans the steps, runs them, writes the manifest; command line |
| `src/daq_commands.py` | builds the **exact Python text** each board will run |
| `src/daq_control.py` | sends that text to the cdaq queue (or to fake boards) and waits for the **receipts** |
| `src/ccd_limits.py` + `configs/limits.toml` | safety checks of every parameter set, voltage ramps |
| `configs/daq_setup.toml` | boards, pool, leader, which boards get θ and are traced, the acmpy command texts |
| `configs/packet_example.toml` | one packet: θ (VSUB 48 V on 101, 50 V on 102), θ_ref (45 V), `[image]`, `[trace]` |
| `src/fake_daq.py` | the fake boards |
| `src/fake_cdaq_monitor.py` | a terminal display of a fake rehearsal, laid out like the real tmux session (section 10) |
| `src/scan_packets.py` | runs many packets (a scan) |

In [1]:
import json, sys, tomllib, shutil, copy
from pathlib import Path
from pprint import pprint

PROJECT = Path.cwd().parent                 # this notebook lives in ccd_opt_ml/notebooks
sys.path.insert(0, str(PROJECT / "src"))
REHEARSAL = PROJECT / "rehearsal"           # everything this notebook writes goes here (safe to delete)

import take_packet as tp
import daq_commands as dc
from ccd_limits import Limits, LimitsError
print("project:", PROJECT)

project: /Users/dylanrosenmerkel/Documents/Software/ccd_opt_ml


## 1. How commands reach a board, and how this follows your scripts

Your scripts never open a board. They put Python text in the **cdaq queue** with
`add2queue(text, target=...)`, and the cdaq client of the target runs it with its own `daq`
(an `acmpy.ACM`):

```
take_packet.py ──add2queue(text, target)──▶ MQTT ──▶ cdaq_main (queue "default")
                                                        │  waits until the target is free
                                                        ▼
                    cdaq client "acm101" / "acm102"   (or both at once: the pool "acmpool")
                                                        │  exec(text) with its own `daq`
                                                        ▼
                                              acmpy ──UDP──▶ acmdaq ──▶ ACM board
```

The packet script sends the same calls as your scripts and command-line routine:

| your routine | what it sends | in a packet |
|---|---|---|
| `cdaq qadd "daq.use_external_clock()" acmpool` | once per session | step `standalone` sends it **only if** a board does not already report the external clock; it is never switched back |
| `cdaq qadd "daq.set_standalone()" acmpool` / `take_trace.py`: `daq.standalone()`, `time.sleep(.1)` | before traces | step `standalone`: `daq.standalone()`, 0.1 s pause |
| `take_trace.py acm101`: `daq.take_trace(fname="trace", subroutine="AcquireImage", EXPOSURE=0, NROW=2, NCOL=5, NSBIN=20, NPBIN=20, NDCM=500, BOOLCLR=0, ch=i)` for i = 0..3 | per board | steps `ref_traces_n` / `traces_n`: the same call for the 4 channels (one queue entry per board, so one receipt lists the 4 files); the 5 cuts are also set to 0 (section 3) |
| `take_image.py -f`: `daq.standalone()`, `daq.flush_ccd()` | flush | steps `flush_xN`: `daq.flush_ccd()` N times |
| `take_image.py -m 101 -f`: `daq.master(101)`, `daq.take_image(fname="img", <every key of the image_params TOML>)` to `acmpool`, then `daq.standalone()`, `daq.flush_ccd()` | the image | step `image`: the same calls with the `[image]` table, as **one** queue entry |
| `cdaq qadd "daq.flush_ccd()" acmpool -t repeat` | idle flushing | the **repeat queue** is saved and cleared during a packet or scan and put back afterwards (section 1b) |
| `set_vdd.py -19` / `daq["VDD"] = -19` in take_image.py | VDD | part of θ (every packet sets every parameter of every board) |

Differences on purpose:

* file name prefixes start with the packet id (`example_00001_ref_traces1_...`), so the files of a
  packet are easy to find;
* every piece of text ends by writing a small **receipt** file (section 6). The queue is
  fire-and-forget, and the receipt is how the script knows a step finished and whether it failed.

### 1b. The repeat queue

A cdaq queue has two parts (`ccd_cdaq/_cdaqqueue.py`): the **simple** queue (each command runs once,
in order) and the **repeat** queue (its commands run again and again). The queue runner
(`ccd_cdaq/_main.py`) works like this:

1. if the simple queue has a command, run it;
2. otherwise run the first repeat command and move it to the back of the repeat list;
3. run **one command at a time**, waiting until the target client is back in standby.

The group keeps `daq.flush_ccd()` (target `acmpool`) in the repeat queue, so the CCDs are flushed
about every 27 s while nothing else is queued. This is the "On repeat queue:
`{'to': 'acmpool', 'cmd': 'daq.flush_ccd()'}`" line in the tmux session.

The packet script queues a step, **waits for its receipt**, then queues the next. So between two
steps the simple queue is empty and the runner starts a repeat flush. The next step then waits
behind it, up to about 27 s. Two consequences:

| problem | fix |
|---|---|
| **extra flushes** between the packet's steps (e.g. between two reference trace sets), which the packet did not ask for and which change the CCD's history | `[daq] repeat_queue = "suspend"` (default): before the first step, the repeat queue is saved to `<work_dir>/queue_backups/` and cleared. It is put back at the end of the packet, or of the whole scan for `scan_packets.py`. After an **error** it is *not* put back, because the boards may still be in leader/follower mode; the backup file lists the `cdaq qadd ... -t repeat` lines to restore it by hand |
| a repeat flush while the boards are in **leader/follower mode** waits for an external start that never comes: the queue hangs | the image step is **one** queue entry: `daq.master(101)` → `take_image` → `daq.standalone()` (in a `finally`) → flush. Nothing can run in between, even with `repeat_queue = "keep"` |

`repeat_queue = "keep"` leaves the repeat queue alone. The packet still works, but with the extra
flushes. The manifest records what was suspended (`repeat_queue_suspended`).

## 2. The DAQ setup file

`configs/daq_setup.toml` says which boards exist and which board does what. It also holds the
**acmpy command texts**, so they can be adapted without changing code.

In [2]:
setup = tp.DaqSetup.load(PROJECT / "configs" / "daq_setup.toml")
for name in ("queue", "pool", "boards", "client", "leader", "theta_boards", "trace_boards",
             "trace_channels", "work_dir", "timeout_margin_s", "fits_wait_s"):
    print(f"{name:16s} = {getattr(setup, name)!r}")
print("\ncommands:")
for name, text in setup.commands.items():
    print(f"  {name:16s} {text!r}")

queue            = 'default'
pool             = 'acmpool'
boards           = [101, 102]
client           = 'acm{board}'
leader           = 101
theta_boards     = [101, 102]
trace_boards     = [101, 102]
trace_channels   = [0, 1, 2, 3]
work_dir         = '/home/damicm/run/ccd_opt_ml'
timeout_margin_s = 300.0
fits_wait_s      = 180.0

commands:
  leader_mode      'daq.master({leader})'
  standalone_mode  'daq.standalone()'
  external_clock   'daq.use_external_clock()'
  flush            'daq.flush_ccd()'
  erase            'daq.erase_ccd(highP=9, wait_for_flush=True)'


What the synchronisation commands do, from `acmpy/fpgacommand.py`:

| command | what the board is told | used for |
|---|---|---|
| `daq.master(101)` on board 101 | `EXS` external start ON + `ESS` send start ON: **leader** | the image |
| `daq.master(101)` on board 102 | `EXS` external start ON + `DSS` send start OFF: **follower** | the image |
| `daq.standalone()` | `DXS` external start OFF + `DSS`: start generated inside the board | flushes, traces |
| `daq.use_external_clock()` | `CEX`: clock from the synchronisation board | set once |

**Why flushes and traces must be standalone.** Both start the sequencer. With external start on,
the sequencer waits for a start pulse that only comes when the *leader* starts an acquisition, so a
trace or a flush would wait forever. This is the "must be in standalone to take PSD or trace" in
your notes.

**`daq.flush_ccd()` blocks until the flush is done.** It runs the sequencer routine `FlushCCD`
(20 rows × [1536 vertical shifts + 6160 horizontal pixels]). acmpy's `start_sequencer(wait=True)`
then sleeps for the routine's duration: about 27 s at the reference timing, plus acmpy's margin.

The **acmpy on the DAQ machine is commit `e750760`**, which is not in my local copy. Your current
scripts use `daq.standalone()`, `daq.flush_ccd()`, `daq.master()`, `daq.take_trace()` and
`daq.take_image()`, so those names are confirmed. `daq.command.get_system_status()` (used to read the
clock) is only in my local copy; if it is missing, the clock command is simply sent (section 5).

## 3. The packet file: θ per board, image size, trace parameters

* `[theta]` holds the parameters shared by every board in `theta_boards`; `[theta_board.102]`
  replaces some of them for board 102 only. The same holds for `[theta_ref]` and
  `[theta_ref_board.<id>]`.
* `[image]` is the image size and readout, like an `image_params` TOML file. It holds every geometry
  key, `acq_level` and `subroutine`.
* `[trace]` holds the trace parameters. Missing keys take your `take_trace.py` values.
* **θ may not contain image or trace sizes.** Every trace and every image sends its complete
  geometry, so both boards always read the same size and nothing is inherited. Example: the 10-06
  image had `NCOLPRE = 5300`; a trace sent right after it without `NCOLPRE` would have read 5300
  prescan columns too. Your 10-08 traces had `NCOLPRE = 0`, so I set the five cuts to 0 explicitly.

**The warning "the traces read their pixels differently from the image"** appears below for this
example packet: its traces use 20×20 binning (as `take_trace.py`) while its image uses 1×10. The
recommended packet (`configs/packet_recommended.toml`, reasons in `docs/packet_design.md`) sets
`[trace] match_image = true`, so the traces read their pixels exactly like the image. That is a 1×1,
500-skip, 16×1000 dark-current image that WADERs can fit, with traces of 2 × 5 pixels after a 100-pixel
prescan. This walkthrough keeps the older example so that its outputs stay comparable.

In [3]:
params = tp.load_toml_or_json(PROJECT / "configs" / "packet_example.toml")
request = tp.request_from_params(params, setup)
print("packet id:", request.packet_id)
for board in setup.theta_boards:
    print(f"acm{board}: {len(request.theta[board])} parameters; change from θ_ref:",
          tp.theta_difference(request.theta[board], request.theta_ref[board]))
print("\nimage:", dc.image_parameters(request.image))
print("trace:", dc.trace_parameters(request.trace))
print("\nsequence:", {k: getattr(request, k) for k in ("n_initial_flushes", "n_reference_traces", "n_traces",
      "flushes_between_traces", "flushes_after_change", "erase_policy", "reference_traces")})

packet id: example_00001
acm101: 33 parameters; change from θ_ref: {'VSUB': 3.0}
acm102: 33 parameters; change from θ_ref: {'VSUB': 5.0}

image: {'subroutine': 'AcquireImage', 'acq_level': 2, 'EXPOSURE': 10, 'NDCM': 500, 'NROW': 100, 'NCOL': 130, 'NPBIN': 1, 'NSBIN': 10, 'NDCMPRE': 0, 'NROWPRE': 0, 'NROWPOST': 0, 'NCOLPRE': 500, 'NCOLPOST': 0, 'BOOLCLR': 1}
trace: {'subroutine': 'AcquireImage', 'EXPOSURE': 0, 'NROW': 2, 'NCOL': 5, 'NSBIN': 20, 'NPBIN': 20, 'NDCM': 500, 'BOOLCLR': 0, 'NDCMPRE': 0, 'NROWPRE': 0, 'NROWPOST': 0, 'NCOLPRE': 0, 'NCOLPOST': 0}

sequence: {'n_initial_flushes': 3, 'n_reference_traces': 3, 'n_traces': 3, 'flushes_between_traces': 1, 'flushes_after_change': 1, 'erase_policy': 'never', 'reference_traces': 'always'}


## 4. Safety checks, before anything is sent

`plan_packet` checks everything **first**. If anything is wrong, it raises before a single command
exists:

1. θ and θ_ref are given for exactly the boards in `theta_boards`, and contain no geometry;
2. every parameter of every board is known and inside its window in `configs/limits.toml`;
3. every voltage is inside what the hardware can produce (bias potentiometer and DAC ranges);
4. sequencer constants are non-negative integers;
5. relationships between parameters, the group's safe-operation rules (`docs/voltage_rules.txt`,
   encoded on 2026-10-14 in `ccd_limits.relational_constraints`): every clock rail HIGH ≥ LOW; VDD > VDRAIN;
   SW_H > OG_H; DG_H > VR; DG_L < VR; OG_L < VR; OG_L ≥ SW_L; every V clock low above every TG level and
   every H clock low below it; H3 lows above SW_H; VDRAIN < DG_L; VDD below VR. They are checked on the
   **complete** configuration: pins a parameter set does not contain take their nominal value (`[nominal]` in
   `limits.toml`, the module's .bcf), so changing one pin cannot break a rule with another unnoticed.
   The hardware ranges follow the ACM manual (`docs/Sequencer Notes.pdf`): VR −13.8…−2, VDD and VDRAIN
   −27.9…−6, VSUB 6.7…88.3 V, clock rails −10…+10 V;
6. the image and trace parameters are inside their windows, and the image defines its full geometry;
7. every intermediate step of every voltage ramp passes the same checks.

When the real backend is used, `limits.toml` must also say `reviewed = true` (it says `false` now).

In [4]:
limits = Limits.load(PROJECT / "configs" / "limits.toml")
print("limits file reviewed:", limits.reviewed)

bad = {**request.theta[102], "VSUB": 75.0, "V1_A_L": 10.5, "DG_1_L": -2.5}   # DG_1_L above VR (-3 V): a rule
print("A bad parameter set for board 102 is refused with every reason listed:")
for problem in limits.violations(bad):
    print("  -", problem)

limits file reviewed: False
A bad parameter set for board 102 is refused with every reason listed:
  - V1_A_L=10.5 V outside hardware range [-10.00, 10.00] V
  - V1_A_L=10.5 above allowed maximum 9.4
  - VSUB=75.0 above allowed maximum 70.0
  - DG_1_L=-2.5 above allowed maximum -3.0
  - V1_A_H=9.9 is below V1_A_L=10.5
  - DG_1_L=-2.5 must be below VR1=-3.0


In [5]:
def refused(change, label):
    # plan a copy of the example packet with one thing changed and print why it is refused
    r = copy.deepcopy(request); change(r)
    try:
        tp.plan_packet(r, setup, limits, None)
        print(label, "-> accepted")
    except (LimitsError, KeyError, ValueError) as err:
        print(label, "->", " ".join(str(err).splitlines()[:2]))

refused(lambda r: r.theta[102].update(VSUB=75.0), "VSUB = 75 V on board 102 only")
refused(lambda r: r.theta[101].update(NCOL=200), "an image size inside theta")
refused(lambda r: r.image.pop("NCOLPRE"), "an image without NCOLPRE")
refused(lambda r: r.trace.update(NPBIN=60), "trace binning 60 (window max 40)")

the traces read their pixels differently from the image: {'NSBIN': (20, 10), 'NPBIN': (60, 1)} (trace, image); set [trace] match_image = true to read them the same way


VSUB = 75 V on board 102 only -> theta of board 102: parameter set rejected:   - VSUB=75.0 above allowed maximum 70.0
an image size inside theta -> theta of board 101 contains ['NCOL']: the image and trace size belong in [image] and [trace], not in theta
an image without NCOLPRE -> "[image] must define the complete image geometry; missing ['NCOLPRE']"
trace binning 60 (window max 40) -> parameter set rejected:   - NPBIN=60 above allowed maximum 40


### Voltage ramps (per board)

A voltage is never changed by more than its `max_step` in one go (VSUB: 5 V; VR and the clock rails:
0.5 V). A larger change is split into equal steps, 0.5 s apart. All voltages of a board move
together, and every intermediate point is checked. Each board ramps from its own previous values.

In [6]:
start = {**request.theta_ref[101], "VSUB": 45.0, "VR": -3.0}
target = {**request.theta_ref[101], "VSUB": 59.0, "VR": -2.0}
for i, step in enumerate(limits.ramp_plan(target, start)):
    print(f"step {i}: VSUB = {step['VSUB']:.2f} V, VR1 = {step['VR1']:.3f} V")
print("\nfirst packet (previous state unknown): applied directly ->", len(limits.ramp_plan(target, None)), "step")

step 0: VSUB = 49.67 V, VR1 = -2.667 V
step 1: VSUB = 54.33 V, VR1 = -2.333 V
step 2: VSUB = 59.00 V, VR1 = -2.000 V

first packet (previous state unknown): applied directly -> 1 step


## 5. The plan: every step and the exact text each board runs

`plan_packet` returns the ordered list of steps. Each step has a **target**: the pool `acmpool`
(every board at once) or one client (`acm101`). It also has the boards expected to answer, the text,
an expected duration and a timeout.

In [7]:
steps = tp.plan_packet(request, setup, limits, prev_theta=None)
print(f"{'#':>2}  {'step':14s} {'target':8s} {'expected':>9s} {'timeout':>8s}  purpose")
for i, s in enumerate(steps):
    print(f"{i:2d}  {s.name:14s} {s.target:8s} {s.expected_s:8.1f}s {s.timeout_s / 60:6.1f}min  {s.purpose}")
print(f"\n{len(steps)} steps, expected {tp.estimate_packet_seconds(steps) / 60:.1f} min "
      "(8 flushes of ~30 s = 3.9 min incl. the one after the image, image 6.3 min, 24 trace "
      "acquisitions; FITS time not included)")

the traces read their pixels differently from the image: {'NSBIN': (20, 10), 'NPBIN': (20, 1)} (trace, image); set [trace] match_image = true to read them the same way


 #  step           target    expected  timeout  purpose
 0  status         acmpool       1.0s    5.0min  read the run directory, the loaded sequencer and the synchronisation/clock bits
 1  standalone     acmpool       1.0s    5.0min  standalone mode for flushes and traces; external clock if not yet set
 2  apply_ref      acm101        1.5s    5.0min  go to the reference parameters (acm101, 1 ramp step(s))
 3  apply_ref      acm102        1.5s    5.0min  go to the reference parameters (acm102, 1 ramp step(s))
 4  flush_x3       acmpool      88.5s    8.0min  initial flushes at the reference
 5  ref_traces_1   acm101        6.8s    5.3min  reference trace set 1 on acm101, one per channel
 6  ref_traces_1   acm102        6.8s    5.3min  reference trace set 1 on acm102, one per channel
 7  flush_x1       acmpool      29.5s    6.0min  flush between trace sets
 8  ref_traces_2   acm101        6.8s    5.3min  reference trace set 2 on acm101, one per channel
 9  ref_traces_2   acm102        6.8

In [8]:
def show(index):
    s = steps[index]
    print(f"--- step {index}: {s.name} -> {s.target} ---")
    print("\n".join(s.lines))
show(0); print(); show(1)

--- step 0: status -> acmpool ---
_r['directory'] = str(daq.acquire.directory)
_r['seqfile'] = str(getattr(daq.sequencer, 'seqfile', ''))
try:
    _r['system_status'] = daq.command.get_system_status()
except Exception as _e:
    _r['system_status'] = 'unavailable: ' + repr(_e)

--- step 1: standalone -> acmpool ---
daq.standalone()
_time.sleep(0.1)
try:
    _clock = daq.command.get_system_status().get('activeclk')
except Exception:
    _clock = None
_r['clock_before'] = _clock
if _clock != 'external':
    daq.use_external_clock()
    _time.sleep(0.5)
_r['directory'] = str(daq.acquire.directory)
_r['seqfile'] = str(getattr(daq.sequencer, 'seqfile', ''))
try:
    _r['system_status'] = daq.command.get_system_status()
except Exception as _e:
    _r['system_status'] = 'unavailable: ' + repr(_e)


**Step `status` (pool, reads only).** Each board records:

* its run directory, where acmpy writes the files (set by `scripts/setup_run.py ... -rd <run_folder>`);
* the sequencer file it has loaded;
* its synchronisation and clock status bits.

The script stops if a board has a different sequencer loaded from the one the packet expects. The
time estimates and the limits assume `reference_v5_pumping.seq`.

**Step `standalone` (pool).** It runs `daq.standalone()` and a 0.1 s pause, as in `take_trace.py`.
Then the board reads its clock. Only if it is **not** already external does it send
`daq.use_external_clock()`, so the clock is set once and never switched back. If the status cannot
be read on this acmpy version, the command is sent anyway, which is harmless when the clock is
already external. The receipt stores `clock_before`, and the manifest lists the boards where the
packet had to set the clock.

In [9]:
show(2)
other = steps[3]
print(f"\n--- step 3: {other.name} -> {other.target}: lines that differ from step 2 ---")
print("\n".join(line for line in other.lines if line not in steps[2].lines) or "(none: same reference)")

--- step 2: apply_ref -> acm101 ---
daq['delay_Va'] = 6000
daq['delay_Vb'] = 6000
daq['delay_Ha'] = 1800
daq['delay_Hb'] = 1800
daq['delay_Hc'] = 1800
daq['V1_A_H'] = 9.9
daq['V1_A_L'] = 7.9
daq['V1_B_H'] = 9.9
daq['V1_B_L'] = 7.7
daq['V2_C_H'] = 9.9
daq['V2_C_L'] = 7.9
daq['V3_A_H'] = 9.9
daq['V3_A_L'] = 7.9
daq['V3_B_H'] = 9.9
daq['V3_B_L'] = 7.7
daq['TG_A_H'] = 7.4
daq['TG_A_L'] = 7.4
daq['TG_B_H'] = 7.4
daq['TG_B_L'] = 7.4
daq['H1_A_H'] = 9.5
daq['H1_A_L'] = 6.7
daq['H1_B_H'] = 9.5
daq['H1_B_L'] = 6.5
daq['H2_C_H'] = 9.0
daq['H2_C_L'] = 6.7
daq['H3_A_H'] = 9.5
daq['H3_A_L'] = 6.7
daq['H3_B_H'] = 9.5
daq['H3_B_L'] = 6.5
daq['VSUB'] = 45.0
daq['VDD1'] = -19.0
daq['VDD2'] = -19.0
daq['VDRAIN1'] = -23.0
daq['VDRAIN2'] = -23.0
daq['VR1'] = -3.0
daq['VR2'] = -3.0
_r['ramp_steps'] = 1

--- step 3: apply_ref -> acm102: lines that differ from step 2 ---
(none: same reference)


**Steps `apply_ref` (one per board).** Each board gets its own text: first the timing constants (they
take effect when acmpy recompiles the sequencer at the next acquisition), then the voltages. Group
keys are already expanded (`VDD` → `VDD1`, `VDD2`), so the text shows exactly which pins are
written. With a known previous state and a large change, the ramp steps appear one after the
other with `_time.sleep(0.5)` between them. Here both boards have the same reference, so the two
texts are identical.

In [10]:
show(4); print(); show(5)

--- step 4: flush_x3 -> acmpool ---
daq.flush_ccd()
daq.flush_ccd()
daq.flush_ccd()

--- step 5: ref_traces_1 -> acm101 ---
daq.take_trace(fname='example_00001_ref_traces1', ch=0, subroutine='AcquireImage', EXPOSURE=0, NROW=2, NCOL=5, NSBIN=20, NPBIN=20, NDCM=500, BOOLCLR=0, NDCMPRE=0, NROWPRE=0, NROWPOST=0, NCOLPRE=0, NCOLPOST=0)
_r['files'].append(str(daq.acquire._filepath))
daq.take_trace(fname='example_00001_ref_traces1', ch=1, subroutine='AcquireImage', EXPOSURE=0, NROW=2, NCOL=5, NSBIN=20, NPBIN=20, NDCM=500, BOOLCLR=0, NDCMPRE=0, NROWPRE=0, NROWPOST=0, NCOLPRE=0, NCOLPOST=0)
_r['files'].append(str(daq.acquire._filepath))
daq.take_trace(fname='example_00001_ref_traces1', ch=2, subroutine='AcquireImage', EXPOSURE=0, NROW=2, NCOL=5, NSBIN=20, NPBIN=20, NDCM=500, BOOLCLR=0, NDCMPRE=0, NROWPRE=0, NROWPOST=0, NCOLPRE=0, NCOLPOST=0)
_r['files'].append(str(daq.acquire._filepath))
daq.take_trace(fname='example_00001_ref_traces1', ch=3, subroutine='AcquireImage', EXPOSURE=0, NROW=2, NCOL=

**Step `flush_x3` (pool).** Three flushes, one after the other, on both boards at once, at the
reference parameters. **No erase and no e-purge**: `erase_policy = "never"`. The erase procedure is
still available as an option, but never the e-purge.

**Steps `ref_traces_1` (`acm101`, then `acm102`).** One `take_trace` per channel, as in your
`take_trace.py`. In acmpy, `take_trace(ch=c, ...)`:

1. sets the given parameters;
2. sets channel *c* to level 0 (every ADC sample) and the other channels to level 3 (nothing);
3. prepares the sequencer and writes the `.meta`, waits 0.5 s, starts the readout (**no exposure**);
4. writes `<prefix>_<NROW>x<NCOL>x<NDCM>_<NPBIN>x<NSBIN>_ch<c>_<board>_<UTC>_<count>.RJ45ADCDATA0..3`.

After each trace, `daq.acquire._filepath` (the file stem acmpy just used) goes into the receipt.

After that come a flush, set 2, a flush, set 3, then `apply` (θ, each board with its own values), a
flush, and the three trace sets at θ.

In [11]:
apply_101 = next(i for i, s in enumerate(steps) if s.name == "apply" and s.target == "acm101")
apply_102 = apply_101 + 1
print("the voltage lines that differ between the two boards' θ:")
for i in (apply_101, apply_102):
    print(f"  {steps[i].target}:", [line for line in steps[i].lines if "VSUB" in line])

the voltage lines that differ between the two boards' θ:
  acm101: ["daq['VSUB'] = 48.0"]
  acm102: ["daq['VSUB'] = 50.0"]


In [12]:
show(len(steps) - 1)

--- step 24: image -> acmpool ---
daq.master(101)
_time.sleep(0.5)
_r['directory'] = str(daq.acquire.directory)
_r['seqfile'] = str(getattr(daq.sequencer, 'seqfile', ''))
try:
    _r['system_status'] = daq.command.get_system_status()
except Exception as _e:
    _r['system_status'] = 'unavailable: ' + repr(_e)
try:
    daq.take_image(fname='example_00001_img', subroutine='AcquireImage', acq_level=2, EXPOSURE=10, NDCM=500, NROW=100, NCOL=130, NPBIN=1, NSBIN=10, NDCMPRE=0, NROWPRE=0, NROWPOST=0, NCOLPRE=500, NCOLPOST=0, BOOLCLR=1)
    _r['files'].append(str(daq.acquire._filepath))
finally:
    daq.standalone()
    _time.sleep(0.1)
    _r['standalone_after_image'] = True
daq.flush_ccd()



**Step `image` (pool): one queue entry from leader mode back to standalone.**

1. `daq.master(101)`: board 101 becomes the leader and the others followers, as
   `take_image.py -m 101` does. The clock is not touched (it is already external). The receipt
   records the status bits afterwards.
2. `take_image` on every board at once, with the complete `[image]` table. In acmpy:
   * every keyword is set (the traces changed `NROW`, `NCOL`, the binning, `BOOLCLR`, ...).
     `acq_level` is always sent, because after a trace the stored level is still the trace's
     `[0, 3, 3, 3]`;
   * the sequencer is prepared, the `.meta` is written, then a 0.5 s wait;
   * **`EXPOSURE` is a Python sleep** (`acquire.start_exposure`); the CCD integrates during it;
   * the sequencer starts; on the followers the readout waits for the leader's start, so both CCDs
     are read out together;
   * the `.meta` goes to the image builder, which writes `avg_img_...fits.bz2`. The script waits for
     that file (`fits_wait_s`) and records it.
3. `daq.standalone()`, inside a `finally`, so it also runs if `take_image` raises. The receipt
   says `standalone_after_image: true`; without it the manifest gets a warning.
4. `daq.flush_ccd()` × `n_final_flushes` (default 1), as `take_image.py -f` does.

Why one entry: between two queue entries the cdaq runner may start a **repeat flush** (section 1b).
In leader/follower mode that flush would hang. Within one entry nothing else can run. After the
image the boards are in standalone mode, so the group's repeat flush is safe again when it is put
back.


## 6. The receipt frame around every step

This is the complete text for one step, exactly as the cdaq client receives it. The client runs it
with `exec(text, globals())`, where `daq` is its board.

In [13]:
print(steps[0].code("00_status", f"{setup.work_dir}/receipts/{request.packet_id}"))

import json as _json, time as _time, traceback as _tb, pathlib as _pl
_r = {'step': '00_status', 'board': daq.id, 't_start': _time.time(), 'files': []}
try:
    _r['directory'] = str(daq.acquire.directory)
    _r['seqfile'] = str(getattr(daq.sequencer, 'seqfile', ''))
    try:
        _r['system_status'] = daq.command.get_system_status()
    except Exception as _e:
        _r['system_status'] = 'unavailable: ' + repr(_e)
    _r['ok'] = True
except Exception:
    _r['ok'] = False
    _r['error'] = _tb.format_exc()
_r['t_end'] = _time.time()
_pl.Path('/home/damicm/run/ccd_opt_ml/receipts/example_00001', '00_status' + '.acm' + str(daq.id) + '.json').write_text(_json.dumps(_r, default=str))



Line by line:

* the imports use private names (`_json`, `_time`, ...) so nothing in the client's namespace is
  overwritten;
* `_r` collects what the step reports: `step`, `board`, start time, `files`;
* the step's own lines run inside `try`. Any exception is caught and its full traceback goes into
  the receipt (otherwise it would only appear in the client log);
* the receipt is written to `<work_dir>/receipts/<packet>/<step>.acm<board>.json`. That directory
  must be **writable by the cdaq clients**; the script creates it before the first step.

The script (`daq_control.wait_for_receipts`) polls for one receipt per expected board:

* every receipt there and `ok` → the next step;
* any receipt with `ok = false` → stop, showing that board's traceback;
* receipts missing after the timeout → stop, saying the **board state is unknown**. Check
  `cdaq print queues` and the client log first. A cdaq client cannot be interrupted during an
  acquisition, so the script sends nothing more.

## 7. Rehearsal: two packets on fake boards

`make_controller("fake", ...)` creates one `FakeDAQ` per board. The controller runs **the same text**
on them that the cdaq clients would run. The fake boards copy real files from `data/`:

* board 101's traces cycle through the five 2026-10-08 repeats of each channel. One ch0 repeat lost
  data, so the automatic retake gets exercised;
* board 102's traces come from the 2026-10-06 run;
* each board's image is the 2026-10-06 100×1000×3000 image.

So the content is real but **not** taken at these settings, and every `.meta` the fake writes says
`"FAKE": true`. The fake boards start on the **internal** clock, to show the clock being set once.

The fake **queue** (`FakeController`) follows the real runner. It starts like the group's session,
with `daq.flush_ccd()` in the repeat queue. Before each submitted step it runs the next repeat
command, because on the real DAQ the runner would have started one while the script waited. Here
the packet runs inside `repeat_queue_suspended`, as `take_packet.py` does from the command line.

In [14]:

from daq_control import repeat_queue_suspended

shutil.rmtree(REHEARSAL, ignore_errors=True)
setup.work_dir = str(REHEARSAL / "work")          # for the rehearsal only
controller = tp.make_controller("fake", setup, fake_data=str(PROJECT / "data"))
print("repeat queue before:", list(controller.repeat))
with repeat_queue_suspended(controller, tp.repeat_queue_backup_file(setup, request.packet_id)) as saved:
    print("repeat queue during the packet:", list(controller.repeat))
    first = tp.take_packet(controller, request, setup, limits, prev_manifest=None,
                           allow_unreviewed_limits=True,      # fake boards: unreviewed limits allowed
                           repeat_queue_suspended=saved)
print("repeat queue after:", list(controller.repeat))
print("packet 1:", first["status"], "| external clock set on boards:", first["external_clock_set_by_packet"])
print("          warnings:", first["warnings"])


the traces read their pixels differently from the image: {'NSBIN': (20, 10), 'NPBIN': (20, 1)} (trace, image); set [trace] match_image = true to read them the same way


repeat queue before: [{'to': 'acmpool', 'cmd': 'daq.flush_ccd()'}]
repeat queue during the packet: []


repeat queue after: [{'to': 'acmpool', 'cmd': 'daq.flush_ccd()'}]
packet 1: ok | external clock set on boards: [101, 102]
          warnings: ['ref_traces_3 on acm101: damaged trace(s), retake 1']


In [15]:
print("What each board was asked to do, in order (parameter writes daq[key] = value left out):")
for board, fake in controller.fakes.items():
    calls = [c for c in fake.calls if c[0] != "set"]
    text = ", ".join(c[0] + (f"({c[1]})" if len(c) > 1 and not isinstance(c[1], dict) else "") for c in calls)
    print(f"\nacm{board}: {text}")

What each board was asked to do, in order (parameter writes daq[key] = value left out):

acm101: standalone, use_external_clock, flush_ccd, flush_ccd, flush_ccd, take_trace(0), take_trace(1), take_trace(2), take_trace(3), flush_ccd, take_trace(0), take_trace(1), take_trace(2), take_trace(3), flush_ccd, take_trace(0), take_trace(1), take_trace(2), take_trace(3), take_trace(0), take_trace(1), take_trace(2), take_trace(3), flush_ccd, take_trace(0), take_trace(1), take_trace(2), take_trace(3), flush_ccd, take_trace(0), take_trace(1), take_trace(2), take_trace(3), flush_ccd, take_trace(0), take_trace(1), take_trace(2), take_trace(3), master(101), take_image(2), standalone, flush_ccd

acm102: standalone, use_external_clock, flush_ccd, flush_ccd, flush_ccd, take_trace(0), take_trace(1), take_trace(2), take_trace(3), flush_ccd, take_trace(0), take_trace(1), take_trace(2), take_trace(3), flush_ccd, take_trace(0), take_trace(1), take_trace(2), take_trace(3), flush_ccd, take_trace(0), take_trace(

Read each list from left to right. The clock is set at the start. Then come 3 flushes, then the
trace sets with flushes between them, then `master`, the image, `standalone` and the final flush
(all four from the one `image` entry). Board 101 shows one extra trace
set: one of its reference sets copied the damaged 10-08 file, so the quality check failed and the
set was retaken (see the warning). `epurge_ccd` never appears; the fake would raise if it were
called.

Now a second packet with the same parameters. Its `prev_manifest` is the first packet, so the boards
ramp from where packet 1 left them, and the clock is **not** set again:

In [16]:

second_request = copy.deepcopy(request); second_request.packet_id = "example_00002"
with repeat_queue_suspended(controller, tp.repeat_queue_backup_file(setup, second_request.packet_id)) as saved:
    second = tp.take_packet(controller, second_request, setup, limits, prev_manifest=first,
                            allow_unreviewed_limits=True, repeat_queue_suspended=saved)
print("packet 2:", second["status"], "| external clock set on boards:", second["external_clock_set_by_packet"])
print("          previous packet:", second["prev_packet_id"], "| seconds since:", second["seconds_since_previous"])
n_clock = {b: sum(1 for c in f.calls if c[0] == "use_external_clock") for b, f in controller.fakes.items()}
print("use_external_clock calls over both packets:", n_clock)


the traces read their pixels differently from the image: {'NSBIN': (20, 10), 'NPBIN': (20, 1)} (trace, image); set [trace] match_image = true to read them the same way


packet 2: ok | external clock set on boards: []
          previous packet: example_00001 | seconds since: 0.0
use_external_clock calls over both packets: {101: 1, 102: 1}


## 8. The manifest

This is the file the processing script starts from. θ, θ_ref, the change and the parameter-set ids
are stored **per board**. Every trace entry has its board, channel and repeat number (1–3).

In [17]:
m = json.loads((REHEARSAL / "work" / "packets" / "example_00001.json").read_text())
short = {k: v for k, v in m.items() if k not in ("theta", "theta_ref", "steps", "trace_quality",
                                                  "reference_traces", "traces", "images", "image", "trace")}
pprint(short, width=110)

{'daq': {'boards': [101, 102],
         'leader': 101,
         'pool': 'acmpool',
         'queue': 'default',
         'theta_boards': [101, 102],
         'trace_boards': [101, 102],
         'trace_channels': [0, 1, 2, 3]},
 'date_end': '2026-10-13T20:34:38.338722',
 'date_start': '2026-10-13T20:34:34.278255',
 'delta_theta': {'101': {'VSUB': 3.0}, '102': {'VSUB': 5.0}},
 'erase_done': [],
 'estimated_seconds': 787,
 'external_clock_set_by_packet': [101, 102],
 'limits_file': '/Users/dylanrosenmerkel/Documents/Software/ccd_opt_ml/configs/limits.toml',
 'limits_reviewed': False,
 'manifest_version': 3,
 'packet_id': 'example_00001',
 'prev_packet_id': None,
 'prev_theta': None,
 'reference_id': {'101': '2a06f2128a', '102': '2a06f2128a'},
 'repeat_queue_suspended': [{'cmd': 'daq.flush_ccd()', 'to': 'acmpool'}],
 'run_directories': {'101': '/Users/dylanrosenmerkel/Documents/Software/ccd_opt_ml/rehearsal/work/fake_run',
                     '102': '/Users/dylanrosenmerkel/Documents/Sof

In [18]:
print("reference traces (board, channel, repeat):")
for t in m["reference_traces"][:5]:
    print(f"  acm{t['board']} ch{t['channel']} set {t['repeat']}: {Path(t['stem']).name}")
print(f"  ... {len(m['reference_traces'])} in total; traces at θ: {len(m['traces'])}")
print("images:")
for im in m["images"]:
    print(f"  acm{im['board']}: {Path(im['stem']).name}  FITS: {[Path(f).name for f in im['fits']]}")
print("\nfirst steps (id, seconds):", [(s["step"], s.get("seconds")) for s in m["steps"][:5]], "...")
damaged = {Path(k).name: v for k, v in m["trace_quality"].items() if v}
print("\ntraces with problems (kept in trace_quality, replaced in the trace list by the retake):")
pprint(damaged)

reference traces (board, channel, repeat):
  acm101 ch0 set 1: example_00001_ref_traces1_2x5x500_20x20_ch0_101_20261013_203434_0
  acm101 ch1 set 1: example_00001_ref_traces1_2x5x500_20x20_ch1_101_20261013_203434_1
  acm101 ch2 set 1: example_00001_ref_traces1_2x5x500_20x20_ch2_101_20261013_203434_2
  acm101 ch3 set 1: example_00001_ref_traces1_2x5x500_20x20_ch3_101_20261013_203434_3
  acm102 ch0 set 1: example_00001_ref_traces1_2x5x500_20x20_ch0_102_20261013_203435_0
  ... 24 in total; traces at θ: 24
images:
  acm101: example_00001_img_100x130x500_1x10_L2_101_20261013_203438_28  FITS: ['avg_example_00001_img_100x130x500_1x10_L2_101_20261013_203438_28.fits.bz2']
  acm102: example_00001_img_100x130x500_1x10_L2_102_20261013_203438_24  FITS: ['avg_example_00001_img_100x130x500_1x10_L2_102_20261013_203438_24.fits.bz2']

first steps (id, seconds): [('00_status', 0.0), ('01_standalone', 0.6), ('02_apply_ref_acm101', 0.0), ('03_apply_ref_acm102', 0.0), ('04_flush_x3', 0.0)] ...

traces with 

**Trace quality check.** After each trace step the script decodes the new traces (`decode_trace.py`;
on the DAQ machine the files are local) and runs `quality_problems()`. This is the check that found
the damaged 2026-10-08 trace, which had lost UDP packets. If a trace in a set is damaged, that
board's whole set is repeated once (`retake_damaged_traces = 1`), and a warning goes into the
manifest. If the check cannot run (numpy missing), the trace is marked `"not checked"` and is never
retaken because of that.

## 9. What the fake boards catch

The fake boards are strict where the real hardware would hang or misbehave silently. Here are five
deliberate mistakes.

In [19]:
from daq_control import StepError, FakeController, wait_for_receipts
from fake_daq import FakeDAQ

def one_step(controller, target, boards, lines, name, timeout=5):
    rdir = REHEARSAL / "mistakes"; rdir.mkdir(parents=True, exist_ok=True)
    controller.submit(target, dc.wrap_step(name, lines, str(rdir)))
    return wait_for_receipts(rdir, name, boards, timeout, 0.2)

def first_and_last_line(err):
    lines = str(err).strip().splitlines()
    return lines[0] + "\n     " + lines[-1]

fakes = {b: FakeDAQ(b, REHEARSAL / "mistakes") for b in (101, 102)}
fc = FakeController(fakes, {"acmpool": [101, 102], "acm101": [101], "acm102": [102]})
trace = dc.trace_parameters({})

# (a) a trace or a flush while the boards are in leader/follower mode
one_step(fc, "acmpool", [101, 102], ["daq.master(101)"], "a_leader")
for name, lines in (("a_trace", dc.trace_lines("x", [0], trace)), ("a_flush", ["daq.flush_ccd()"])):
    try:
        one_step(fc, "acm101", [101], lines, name)
    except StepError as err:
        print("(a)", first_and_last_line(err))

(a) step a_trace failed on board(s) [101]:
     fake_daq.FakeHangError: acm101: trace in leader mode -- the sequencer would wait for an external start that never comes
(a) step a_flush failed on board(s) [101]:
     fake_daq.FakeHangError: acm101: flush in leader mode -- the FlushCCD routine would wait for an external start


In [20]:
# (b) an image right after a trace without acq_level (board 101 is still at the trace's level)
one_step(fc, "acmpool", [101, 102], ["daq.standalone()"], "b_standalone")
one_step(fc, "acm101", [101], dc.trace_lines("x", [0], trace), "b_trace")
one_step(fc, "acmpool", [101, 102], ["daq.master(101)"], "b_leader")
without_level = {k: v for k, v in dc.image_parameters(request.image).items() if k != "acq_level"}
try:
    one_step(fc, "acmpool", [101, 102], [f"daq.take_image(fname='x', {dc._kwargs(without_level)})"], "b_image")
except StepError as err:
    print("(b)", first_and_last_line(err))

(b) step b_image failed on board(s) [101]:
     fake_daq.FakeHangError: acm101: image with acquisition levels [0, 3, 3, 3]; pass acq_level=1 or 2 (after a trace the level is still the trace's)


In [21]:
# (c) a board that never answers (queue paused, client offline, acquisition hung)
class SilentController(FakeController):
    def submit(self, target, code):
        self.sent.append((target, code))     # accepted by the queue, never executed
try:
    one_step(SilentController(fakes, fc.targets), "acmpool", [101, 102], ["pass"], "c_lost", timeout=2)
except StepError as err:
    print("(c)", err)

(c) step c_lost: no receipt from board(s) [101, 102] after 2 s. The board state is UNKNOWN: check `cdaq print queues` and the client log before doing anything else.


In [22]:
# (d) a parameter set outside the limits: refused before ANY text is sent
fc2 = FakeController(fakes, fc.targets)
bad_request = copy.deepcopy(request); bad_request.theta[102]["VSUB"] = 70.0
try:
    tp.take_packet(fc2, bad_request, setup, limits, None, allow_unreviewed_limits=True)
except LimitsError as err:
    print("(d)", " ".join(str(err).splitlines()[:2]))
print("    texts sent to the boards:", len(fc2.sent))

(d) theta of board 102: parameter set rejected:   - VSUB=70.0 above allowed maximum 60.0
    texts sent to the boards: 0


In [23]:
# (e) the old packet layout: leader mode as its own queue entry, with the group's repeat flush kept.
#     Between the 'leader' entry and the next one the runner starts the repeat flush -- in
#     leader/follower mode.  On the real DAQ the queue would hang until the step's timeout.
fakes_e = {b: FakeDAQ(b, REHEARSAL / "mistakes") for b in (101, 102)}
fc_e = FakeController(fakes_e, fc.targets, repeat=[{"to": "acmpool", "cmd": "daq.flush_ccd()"}])
one_step(fc_e, "acmpool", [101, 102], ["daq.standalone()"], "e_standalone")
one_step(fc_e, "acmpool", [101, 102], ["daq.master(101)"], "e_leader")
try:
    one_step(fc_e, "acmpool", [101, 102], ["daq.standalone()"], "e_next")
except StepError as err:
    print("(e)", str(err).replace("; ", ";\n     "))

(e) board(s) [101, 102] are hung (acm101: repeat command 'daq.flush_ccd()' failed: FakeHangError: acm101: flush in leader mode -- the FlushCCD routine would wait for an external start;
     acm102: repeat command 'daq.flush_ccd()' failed: FakeHangError: acm102: flush in follower mode -- the FlushCCD routine would wait for an external start). On the real DAQ this step would wait for its full timeout and then stop with 'no receipt'.


## 10. Watching a rehearsal like the real tmux session

On the DAQ machine a tmux session shows `cdaq print queues` (queue: current command, simple and
repeat queue), `cdaq print logs` (board and cdaq log lines, last warning at the bottom),
`cdaq print clients` (each client's status and current command, the pools) and the operator's
shell with the `Publishing: ...` line of every queued command.

The fake controller writes the same information to `<work_dir>/fake_cdaq_state.json` after every
change, and `src/fake_cdaq_monitor.py` draws it in the same layout and formats (taken from
`ccd_cdaq/monitoring.py`). **It is only a display of the fake**: MQTT and the real cdaq are still
untested until the script runs on the DAQ machine.

In two terminals (the rehearsal slowed to 5 % of real time, so a flush takes 1.3 s):

```bash
python src/fake_cdaq_monitor.py --state /tmp/rehearsal/fake_cdaq_state.json
python src/take_packet.py --params configs/packet_example.toml --backend fake --work-dir /tmp/rehearsal --fake-time-scale 0.05
```

or, where tmux is installed (the DAQ machine; on a Mac `brew install tmux`), one tmux session laid
out like the real one, with the rehearsal in the bottom pane:

```bash
python src/fake_cdaq_monitor.py --state /tmp/rehearsal/fake_cdaq_state.json --tmux --run "python src/take_packet.py --params configs/packet_example.toml --backend fake --work-dir /tmp/rehearsal --fake-time-scale 0.05"
```

Here is a snapshot (`--once`) taken 4 s into a packet slowed down to 2 % of real time, during the
reference trace sets. The current command is shown with its step id (`[09_ref_traces_2_acm102]`)
instead of the long receipt frame. The repeat queue is empty because the packet suspended it.

In [24]:
import subprocess, time
live = REHEARSAL / "live"
shutil.rmtree(live, ignore_errors=True)
run = subprocess.Popen([sys.executable, str(PROJECT / "src" / "take_packet.py"), "--params",
                        str(PROJECT / "configs" / "packet_example.toml"), "--backend", "fake",
                        "--work-dir", str(live), "--fake-data", str(PROJECT / "data"),
                        "--fake-time-scale", "0.02"],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)                                         # somewhere in the reference traces
snapshot = subprocess.run([sys.executable, str(PROJECT / "src" / "fake_cdaq_monitor.py"), "--state",
                           str(live / "fake_cdaq_state.json"), "--once", "--width", "190", "--height", "30"],
                          capture_output=True, text=True).stdout
print(snapshot)
run.wait()
print("rehearsal finished with exit code", run.returncode)

printqueue execution time: 2026-10-13 16:34:48   [FAKE]  │example_00001_ref_traces2_2x5x500_20x20_ch2_101_20261013_203447_6         │Client status on 2026-10-13 16:34:48   [FAKE]            
                                                         │2026-10-13 16:34:47.833 - acm.101.sequencer - INFO     - Setting          │Clients:                                                 
[0] default:                                             │subroutine to AcquireImage (1 sec)                                        │[0] acm102: Busy; Current: [09_ref_traces_2_acm102]      
Currently running: {'to': 'acm102', 'cmd':               │2026-10-13 16:34:47.862 - acm.101.acquire - INFO     - Trace ch3 written: │daq.take_trace(fname='example_00001_ref_traces2', ch=0,  
"[09_ref_traces_2_acm102]                                │example_00001_ref_traces2_2x5x500_20x20_ch3_101_20261013_203447_7         │subroutine='Ac ...; Pool: acmpool (0.1sec)               
daq.take_trace(fname='example_00001_ref_trace

rehearsal finished with exit code 0


And the end of the queue's life in mistake (e): the monitor shows both clients **Busy** with the
hung repeat flush, the stuck command still "currently running", the queued step waiting, and the
error at the bottom of the log pane. This is how such a hang would look in the real session:
`cdaq print clients` would show the clients busy with `daq.flush_ccd()` indefinitely.

In [25]:
import fake_cdaq_monitor as mon
fc_e.state_file = REHEARSAL / "mistakes" / "state.json"; fc_e._write_state()
state = json.loads(fc_e.state_file.read_text())
print(mon.draw_all(state, time.time(), width=190, height=26))

printqueue execution time: 2026-10-13 16:35:02   [FAKE]  │subroutine to FlushCCD (26 sec)                                           │Client status on 2026-10-13 16:35:02   [FAKE]            
                                                         │2026-10-13 16:34:44.132 - acm.102.sequencer - INFO     - Setting          │Clients:                                                 
[0] default:                                             │subroutine to FlushCCD (26 sec)                                           │[0] acm102: Busy; Current: daq.flush_ccd(); Pool: acmpool
Currently running: {'to': 'acmpool', 'cmd':              │2026-10-13 16:34:44.132 - acm.101.command - INFO     - Synchronisation:   │(18.5sec)                                                
'daq.flush_ccd()'} (elapsed: 18.5sec)                    │leader (master = acm101)                                                  │[1] acm101: Busy; Current: daq.flush_ccd(); Pool: acmpool
On simple queue:                             

## 11. A scan

`scan_packets.py` builds the list of packets from `configs/scan_example.toml`:

* the reference, then one-at-a-time changes in a seeded random order;
* a design parameter changes every board (`delay_Ha`) or one board only (`acm101.VSUB`,
  `acm102.VR`);
* the reference is repeated after every 5 points;
* every packet has the scan's reference as θ_ref.

The dry run plans **every** packet with all the checks above before the first command. A real run
stops at the first failure, and a rerun skips the packets whose manifest says `ok`.

The repeat queue is suspended **once for the whole scan** (not per packet), so no flush the
packets did not ask for runs between their steps, and it is put back after the last packet.

In [26]:
import scan_packets as sp
config = tomllib.loads((PROJECT / "configs" / "scan_example.toml").read_text())
plan = sp.build_plan(config, setup)
ok = sp.validate_plan(plan, limits, config, setup)

the traces read their pixels differently from the image: {'NSBIN': (20, 10), 'NPBIN': (20, 1)} (trace, image); set [trace] match_image = true to read them the same way


the traces read their pixels differently from the image: {'NSBIN': (20, 10), 'NPBIN': (20, 1)} (trace, image); set [trace] match_image = true to read them the same way


the traces read their pixels differently from the image: {'NSBIN': (20, 10), 'NPBIN': (20, 1)} (trace, image); set [trace] match_image = true to read them the same way


the traces read their pixels differently from the image: {'NSBIN': (20, 10), 'NPBIN': (20, 1)} (trace, image); set [trace] match_image = true to read them the same way


the traces read their pixels differently from the image: {'NSBIN': (20, 10), 'NPBIN': (20, 1)} (trace, image); set [trace] match_image = true to read them the same way


the traces read their pixels differently from the image: {'NSBIN': (20, 10), 'NPBIN': (20, 1)} (trace, image); set [trace] match_image = true to read them the same way


the traces read their pixels differently from the image: {'NSBIN': (20, 10), 'NPBIN': (20, 1)} (trace, image); set [trace] match_image = true to read them the same way


the traces read their pixels differently from the image: {'NSBIN': (20, 10), 'NPBIN': (20, 1)} (trace, image); set [trace] match_image = true to read them the same way


the traces read their pixels differently from the image: {'NSBIN': (20, 10), 'NPBIN': (20, 1)} (trace, image); set [trace] match_image = true to read them the same way


the traces read their pixels differently from the image: {'NSBIN': (20, 10), 'NPBIN': (20, 1)} (trace, image); set [trace] match_image = true to read them the same way


OK  first_scan_00000  ~ 13.1 min  25 steps  change {'acm101': {}, 'acm102': {}}
OK  first_scan_00001  ~ 13.2 min  25 steps  change {'acm101': {'delay_Va': 1500}, 'acm102': {'delay_Va': 1500}}
OK  first_scan_00002  ~ 13.1 min  25 steps  change {'acm101': {}, 'acm102': {'VR': -0.5}}
OK  first_scan_00003  ~ 13.6 min  25 steps  change {'acm101': {'delay_Ha': 600}, 'acm102': {'delay_Ha': 600}}
OK  first_scan_00004  ~ 13.1 min  25 steps  change {'acm101': {'VSUB': 3.0}, 'acm102': {}}
OK  first_scan_00005  ~ 13.1 min  25 steps  change {'acm101': {}, 'acm102': {'VR': 0.5}}
OK  first_scan_00006  ~ 13.1 min  25 steps  change {'acm101': {}, 'acm102': {}}
OK  first_scan_00007  ~ 12.6 min  25 steps  change {'acm101': {'delay_Ha': -600}, 'acm102': {'delay_Ha': -600}}
OK  first_scan_00008  ~ 13.1 min  25 steps  change {'acm101': {'VSUB': -3.0}, 'acm102': {}}
OK  first_scan_00009  ~ 13.0 min  25 steps  change {'acm101': {'delay_Va': -1500}, 'acm102': {'delay_Va': -1500}}

10 packets, estimated total 2

## 12. Before running on the DAQ machine

**A. Check against the acmpy installed there** (commit `e750760`). Your scripts confirm
`standalone()`, `flush_ccd()`, `master()`, `take_trace()`, `take_image()` and
`use_external_clock()`/`set_standalone()`. Still to check:

| used by the script | check |
|---|---|
| `daq.acquire._filepath`, `daq.acquire.directory`, `daq.sequencer.seqfile` | attribute names (used in the receipts) |
| `daq.command.get_system_status()` | exists (if not: the clock command is just sent each packet, and the status is recorded as "unavailable") |
| `take_trace`/`take_image` accept `subroutine`, `acq_level` and the cut keywords | yes in your scripts and in my copy |
| image builder | writes `avg_<image>.fits.bz2` next to the raw files (the 10-06 run did) |

**B. Check the cdaq setup:**

* the pool name `acmpool` (your default in take_image.py), and that acm101 and acm102 are in it and
  attached to the queue;
* the **repeat queue**: with `repeat_queue = "suspend"` the script saves it with `ccd_cdaq.queue2file`
  (the cdaq main process writes the file, so `work_dir/queue_backups/` must be writable by it),
  clears it and adds it back afterwards. Check with `cdaq print queues` that it is back after the
  first packet. After a failed packet put it back by hand (the lines are in the backup file);
* `work_dir` is writable by the user running the clients.

**C. Decide:**

* the trace parameters: they follow your `take_trace.py` (20×20 binning, 500 skips), not the image's
  1×10 binning;
* review `limits.toml`, add the bias/clock relationships, and set `reviewed = true`.

**D. First contact, in this order:**

1. `python take_packet.py --params <packet> --show-commands`: read every text.
2. `python take_packet.py --params <packet> --backend cdaq --status-only`: sends only the read-only
   status step and prints what each board reports (run directory, sequencer, sync and clock bits).
   This proves the queue, the pool, the clients and the receipt directory work.
3. One packet with θ = θ_ref, watching the client logs.
4. Process it (second notebook). Its 3 reference sets and 3 trace sets are 6 repeats at identical
   settings: the first real measurement of the run-to-run scatter.
5. Only then a short scan.